# MecâniQA - Encontro 5 (16/09/2026)

Validação temporal (Time Series Split) e boletim de notas dos baselines.

In [ ]:
import pandas as pd
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
)
from sklearn.model_selection import TimeSeriesSplit

In [ ]:
df = pd.read_csv("data/mecaniqa_dataset.csv")
df["Data"] = pd.to_datetime(df["Data"])
df = df.set_index("Data").sort_index()

serie = df["Trocas_Oleo"].interpolate(method="time")

## Recortes cronológicos

`TimeSeriesSplit` treina no passado e testa no futuro. Não embaralha os dias.

In [ ]:
n_splits = 5
janela = 7

tscv = TimeSeriesSplit(n_splits=n_splits)

for i, (idx_treino, idx_teste) in enumerate(tscv.split(serie), start=1):
    print(
        f"Fold {i}: treino {serie.index[idx_treino[0]].date()} -> "
        f"{serie.index[idx_treino[-1]].date()} "
        f"({len(idx_treino)} dias) | teste {serie.index[idx_teste[0]].date()} -> "
        f"{serie.index[idx_teste[-1]].date()} ({len(idx_teste)} dias)"
    )

## Bloco avaliador (MAE, RMSE, MAPE)

In [ ]:
def previsao_naive(serie):
    return serie.shift(1)


def previsao_media_movel(serie, janela=7):
    return serie.shift(1).rolling(janela).mean()


def calcular_metricas(y_real, y_pred):
    mae = mean_absolute_error(y_real, y_pred)
    rmse = mean_squared_error(y_real, y_pred) ** 0.5
    # MAPE explode se o real for 0 (divisão por zero)
    sem_zero = y_real != 0
    mape = mean_absolute_percentage_error(y_real[sem_zero], y_pred[sem_zero]) * 100
    return mae, rmse, mape


def avaliar_nas_janelas(serie, y_pred, tscv):
    maes, rmses, mapes = [], [], []

    for _, idx_teste in tscv.split(serie):
        real = serie.iloc[idx_teste]
        pred = y_pred.iloc[idx_teste]
        valido = real.notna() & pred.notna()
        mae, rmse, mape = calcular_metricas(real[valido], pred[valido])
        maes.append(mae)
        rmses.append(rmse)
        mapes.append(mape)

    return sum(maes) / len(maes), sum(rmses) / len(rmses), sum(mapes) / len(mapes)

## Boletim dos baselines

In [ ]:
naive = previsao_naive(serie)
mm7 = previsao_media_movel(serie, janela=janela)

mae_n, rmse_n, mape_n = avaliar_nas_janelas(serie, naive, tscv)
mae_m, rmse_m, mape_m = avaliar_nas_janelas(serie, mm7, tscv)

print("Naive")
print(f"Resultados do Baseline - MAE: {mae_n:.2f}, RMSE: {rmse_n:.2f}, MAPE: {mape_n:.2f}%")
print("Medias Moveis (7 dias)")
print(f"Resultados do Baseline - MAE: {mae_m:.2f}, RMSE: {rmse_m:.2f}, MAPE: {mape_m:.2f}%")